# Kaggle runner — PA4 Task 1: Pruning and Quantization

Runs every `task1_compression/notebooks/*.ipynb` in order (papermill, fresh kernel each), saving the **executed notebooks,
results, figures and checkpoints** inside `/kaggle/working/ATML-PA4`, which becomes this run's Output.

**Before running (right-hand panel → Settings):** Accelerator = **GPU T4 x1**, Internet = **On**.
Then use **Save Version → Save & Run All (Commit)** so it keeps running in the background (12 h limit).
Expected time: ~3-4 h (first run also streams C4, ~30-60 min).

**Resuming / splitting across runs:** every notebook skips work whose results already exist. To continue a previous
run, open that run's page → *Output* → **New Dataset** (or "Add Input" → Your Work → that notebook's output), then set
`RESUME_FROM` below to the `ATML-PA4` folder inside it, and trim `NOTEBOOKS` to the ones still to run.

In [ ]:
# ---------------------------------------------------------------- configuration
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"
BRANCH = "main"
TASK = "task1_compression"
NOTEBOOKS = [
    "1_setup_and_fp16_baseline.ipynb",
    "2_pruning_P1_magnitude_P2_wanda.ipynb",
    "3_nonuniform_pruning_P3.ipynb",
    "4_quantization_Q1_uniform_Q2_awq.ipynb",
    "5_mixed_precision_Q3.ipynb",
    "6_structured_2to4_and_tradeoffs.ipynb",
]
RESUME_FROM = None   # e.g. "/kaggle/input/pa4-task1-run1/ATML-PA4"
STOP_ON_ERROR = True  # later notebooks depend on earlier ones

In [ ]:
import os, sys, subprocess, shutil, time
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
ROOT = os.path.join(WORK, "ATML-PA4")
os.chdir(WORK)
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
# pinned versions (manual) + runner deps; the per-notebook kernels start fresh, so no restart is needed here
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                "datasets>=3,<5", "peft>=0.13", "papermill"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------------------------------------------------------- restore a previous run (optional)
if RESUME_FROM:
    for sub in (f"{TASK}/results", f"{TASK}/checkpoints", f"{TASK}/splits", f"{TASK}/notebooks", "atml-pa4-datasets"):
        src, dst = os.path.join(RESUME_FROM, sub), os.path.join(ROOT, sub)
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("restored", sub)

In [ ]:
# ---------------------------------------------------------------- run the task notebooks in order
import papermill as pm
log = []
for nb in NOTEBOOKS:
    path = os.path.join(ROOT, TASK, "notebooks", nb)
    t0 = time.time()
    print()
    print(f"========== RUNNING {nb} ==========", flush=True)
    try:
        pm.execute_notebook(path, path, cwd=os.path.dirname(path), kernel_name="python3",
                            log_output=True, progress_bar=False, request_save_on_cell_execute=True)
        status = "ok"
    except Exception as e:
        status = f"FAILED: {type(e).__name__}"
        print(e)
    mins = (time.time() - t0) / 60
    log.append((nb, status, round(mins, 1)))
    print(f"========== {nb}: {status} ({mins:.1f} min) ==========", flush=True)
    if status != "ok" and STOP_ON_ERROR:
        break
for row in log:
    print(row)

In [ ]:
# ---------------------------------------------------------------- small download bundle (results + executed notebooks)
import zipfile
bundle = os.path.join(WORK, f"{TASK}_results_and_notebooks.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("results", "notebooks", "splits"):
        base = os.path.join(ROOT, TASK, sub)
        for dp, _, files in os.walk(base):
            for f in files:
                full = os.path.join(dp, f)
                z.write(full, os.path.relpath(full, ROOT))
print("wrote", bundle, "(checkpoints stay under ATML-PA4/ in the Output for resuming)")